# **Data Collection**

## Objectives

* Fetch data from Kaggle and save as raw data

## Inputs

* Kaggle JSON file - authentication token

## Outputs

* Generate Dtatset: input/datasets/cherry_leaves_dataset

## Additional Comments

* No comments 



---

# Change working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [12]:
import numpy
import os

RuntimeError: CPU dispatcher tracer already initlized

In [16]:
current_dir = os.getcwd()
current_dir

'/workspaces/project_5_mildew_detection'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [17]:
os.chdir('/workspaces/project_5_mildew_detection')
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [18]:
current_dir = os.getcwd()
current_dir

'/workspaces/project_5_mildew_detection'

# Install Kaggle

Install Kaggle package

In [19]:
%pip install kaggle --no-user

Note: you may need to restart the kernel to use updated packages.


Change Kaggle configuration to current working directory and permission of kaggle authentication json

In [20]:
os.environ['KAGGLE_CONFIG_DIR'] = os.getcwd()
! chmod 600 kaggle.json

Set Kaggle Dataset and Download it

In [21]:
KaggleDatasetPath = "codeinstitute/cherry-leaves"
DestinationFolder = "inputs/cherry_leaves_dataset"
! kaggle datasets download -d {KaggleDatasetPath} -p {DestinationFolder}

if os.path.exists(DestinationFolder):
    print("Files in folder:")
    for f in os.listdir(DestinationFolder):
        print("-", f)
else:
    print("❌ Folder does not exist:", DestinationFolder)

Dataset URL: https://www.kaggle.com/datasets/codeinstitute/cherry-leaves
License(s): unknown
  0%|                                               | 0.00/55.0M [00:00<?, ?B/s]
100%|██████████████████████████████████████| 55.0M/55.0M [00:00<00:00, 1.33GB/s]
Files in folder:
- train
- test
- validation
- cherry-leaves
- cherry-leaves.zip


Unzip the downloaded file, delete the zip file

In [22]:
import zipfile
DestinationFolder = "inputs/cherry_leaves_dataset"
zip_path = os.path.join(DestinationFolder, "cherry-leaves.zip")

if os.path.exists(zip_path):
    print(f"Extracting: {zip_path}")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(DestinationFolder)
    os.remove(zip_path)
    print("✅ Extraction complete and zip file removed.")
else:
    print(f"❌ Zip file not found at: {zip_path}")

Extracting: inputs/cherry_leaves_dataset/cherry-leaves.zip


✅ Extraction complete and zip file removed.


---

# Data Preparation

Data cleaning

Check and remove non-image files

In [23]:
def remove_non_image_file(my_data_dir):
    image_extension = ('.png', '.jpg', '.jpeg')
    folders = os.listdir(my_data_dir)
    for folder in folders:
        files = os.listdir(my_data_dir + '/' + folder)
        # print(files)
        i = []
        j = []
        for given_file in files:
            if not given_file.lower().endswith(image_extension):
                file_location = my_data_dir + '/' + folder + '/' + given_file
                os.remove(file_location)  # remove non image file
                i.append(1)
            else:
                j.append(1)
                pass
        print(f"Folder: {folder} - has image file", len(j))
        print(f"Folder: {folder} - has non-image file", len(i))

In [24]:
remove_non_image_file(my_data_dir='inputs/cherry_leaves_dataset/cherry-leaves')

Folder: powdery_mildew - has image file 2104
Folder: powdery_mildew - has non-image file 0
Folder: healthy - has image file 2104
Folder: healthy - has non-image file 0


# Split train validation test set

In [25]:
import os
import shutil
import random
import joblib


def split_train_validation_test_images(my_data_dir, train_set_ratio, validation_set_ratio, test_set_ratio):

    if train_set_ratio + validation_set_ratio + test_set_ratio != 1.0:
        print("train_set_ratio + validation_set_ratio + test_set_ratio should sum to 1.0")
        return

    # gets classes labels
    labels = os.listdir(my_data_dir)  # it should get only the folder name
    if 'test' in labels:
        pass
    else:
        # create train, test folders with classes labels sub-folder
        for folder in ['train', 'validation', 'test']:
            for label in labels:
                os.makedirs(name=my_data_dir + '/' + folder + '/' + label)

        for label in labels:

            files = os.listdir(my_data_dir + '/' + label)
            random.shuffle(files)

            train_set_files_qty = int(len(files) * train_set_ratio)
            validation_set_files_qty = int(len(files) * validation_set_ratio)

            count = 1
            for file_name in files:
                if count <= train_set_files_qty:
                    # move a given file to the train set
                    shutil.move(my_data_dir + '/' + label + '/' + file_name,
                                my_data_dir + '/train/' + label + '/' + file_name)

                elif count <= (train_set_files_qty + validation_set_files_qty):
                    # move a given file to the validation set
                    shutil.move(my_data_dir + '/' + label + '/' + file_name,
                                my_data_dir + '/validation/' + label + '/' + file_name)

                else:
                    # move given file to test set
                    shutil.move(my_data_dir + '/' + label + '/' + file_name,
                                my_data_dir + '/test/' + label + '/' + file_name)

                count += 1

            os.rmdir(my_data_dir + '/' + label)

In [ ]:
split_train_validation_test_images(my_data_dir=f"/workspaces/project_5_mildew_detection/inputs/cherry_leaves_dataset",
                                train_set_ratio=0.7,
                                validation_set_ratio=0.1,
                                test_set_ratio=0.2
                                )


---